In [174]:
import os
import io
import pandas as pd
import yfinance as yf
import feedparser
import pandas_ta as ta
from typing import List, Optional, Dict, Any, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool

load_dotenv()

True

In [175]:
NEWS_DATA_API = os.getenv("NEWS_DATA_API")


In [176]:
model = ChatOllama(model="gemma4:e4b", temperature=0.3, num_ctx=8192)


In [177]:
def get_valid_nse_tickers() -> set[str]:
    """Downloads official active NSE equity list with a safe fallback."""
    url = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }
    try:
        response = requests.get(url, headers=headers, timeout=8)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))
        valid_symbols = set(df["SYMBOL"].str.strip().str.upper())
        print(f"[Init] Loaded {len(valid_symbols)} active NSE tickers from official exchange list.")
        return valid_symbols
    except Exception as e:
        print(f"[Init Warning] Could not fetch NSE master list ({e}). Using core fallback tickers.")
        return {
            "RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK", "ITC", 
            "SBIN", "BHARTIARTL", "TATAMOTORS", "LICI", "LT", "WIPRO"
        }

VALID_NSE_TICKERS = get_valid_nse_tickers()


[Init] Loaded 2593 active NSE tickers from official exchange list.


In [178]:
import feedparser
import requests
from langchain_core.tools import tool

# @tool
def fetch_free_market_news() -> list:
    """Fetches the latest breaking Indian stock market news for free via RSS."""
    
    # Moneycontrol Top News & Corporate Actions RSS feeds
    rss_urls = [
        "https://www.moneycontrol.com/rss/buzzingstocks.xml", # Stocks in action
        "https://www.moneycontrol.com/rss/results.xml",       # Quarterly Earnings
        "https://www.livemint.com/rss/companies"              # Livemint Company Specific News
    ]
    
    clean_articles = []
    
    # Send a standard Google Chrome user-agent to bypass the 403 Forbidden block
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    for url in rss_urls:
        try:
            # 1. Fetch the XML securely mimicking a browser
            response = requests.get(url, headers=headers, timeout=10)
            
            # 2. Only parse if the request was successful
            if response.status_code == 200:
                feed = feedparser.parse(response.text)
                
                # Grab top 10 articles from each feed
                for entry in feed.entries[:30]:
                    clean_articles.append({
                        "title": entry.title,
                        # Some RSS feeds omit description, so we fallback to title
                        "description": getattr(entry, 'description', entry.title),
                        "source": "Moneycontrol"
                    })
            else:
                print(f"[Warning] Blocked by {url} - Status Code: {response.status_code}")
                
        except Exception as e:
            print(f"[Error] Failed fetching {url}: {e}")
            
    return clean_articles

In [179]:
def fetch_market_news(topic: str = "Indian Stock Market", max_results: int = 10) -> List[Dict[str, str]]:
    """
    Fetches news using flexible catalyst keywords.
    Automatically falls back to top business news if the specific query returns 0 results.
    """
    if not NEWS_DATA_API:
        print("[Error] NEWS_DATA_API environment variable is missing.")
        return []

    # Broad catalyst query avoids over-filtering
    catalyst_query = "deal OR profit OR order OR acquisition OR quarterly OR results"
    base_url = "https://newsdata.io/api/1/latest"
    
    primary_params = {
        "apikey": NEWS_DATA_API,
        "country": "in",
        "category": "business",
        "q": catalyst_query,
        "size": max_results
    }

    try:
        res = requests.get(base_url, params=primary_params, timeout=10)
        data = res.json()
        articles_raw = data.get("results", [])

        # Fallback if specific search returns zero results
        if not articles_raw or data.get("status") != "success":
            print("[News] Specific catalyst query returned 0 articles. Falling back to top business feed...")
            fallback_params = {
                "apikey": NEWS_DATA_API,
                "country": "in",
                "category": "business",
                "size": max_results
            }
            res = requests.get(base_url, params=fallback_params, timeout=10)
            data = res.json()
            articles_raw = data.get("results", [])

        clean_articles = []
        for art in articles_raw:
            title = art.get("title")
            desc = art.get("description")
            if title and desc:
                clean_articles.append({
                    "title": title.strip(),
                    "description": desc.strip(),
                    "source": art.get("source_name", "Financial Media")
                })

        print(f"[News] Successfully loaded {len(clean_articles)} articles.")
        return clean_articles

    except Exception as e:
        print(f"[News Error] Failed to retrieve news: {e}")
        return []


In [180]:
def calculate_ticker_technicals(ticker: str) -> Optional[Dict[str, Any]]:
    """
    Fetches technical metrics for a verified NSE ticker.
    Returns None safely if yfinance has no history or fails (e.g., IPO/delisted).
    """
    base_ticker = ticker.strip().upper().replace(".NS", "").replace(".BO", "")

    if base_ticker not in VALID_NSE_TICKERS:
        return None

    ticker_ns = f"{base_ticker}.NS"
    try:
        stock = yf.Ticker(ticker_ns)
        df = stock.history(period="6mo")

        if df.empty or len(df) < 30:
            print(f"[Technicals] Incomplete trading bars (<30) for {ticker_ns}. Proceeding without technicals.")
            return None

        # Add indicators
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)
        df.ta.ema(length=20, append=True)

        last = df.iloc[-1]
        return {
            "close": round(float(last.get("Close", 0.0)), 2),
            "rsi": round(float(last.get("RSI_14", 0.0)), 2) if not pd.isna(last.get("RSI_14")) else None,
            "ema20": round(float(last.get("EMA_20", 0.0)), 2) if not pd.isna(last.get("EMA_20")) else None,
            "macd": round(float(last.get("MACD_12_26_9", 0.0)), 2) if not pd.isna(last.get("MACD_12_26_9")) else None,
        }
    except Exception as e:
        print(f"[Technicals] Failed to compute metrics for {ticker_ns}: {e}")
        return None


In [181]:
class ExtractedEntity(BaseModel):
    category: Literal["SINGLE_STOCK", "MACRO_SECTOR", "NOISE"] = Field(
        description="Classify whether the article covers an individual listed company, broad economic/market trends, or non-financial noise."
    )
    ticker: str = Field(
        description="Official NSE ticker if SINGLE_STOCK, otherwise 'NONE'."
    )
    news_title: str
    description: str
    source: Optional[str] = Field(default="Unknown", description="The source of the news article")

class IdentifiedArticles(BaseModel):
    items: List[ExtractedEntity]
    
class FinancialTerm(BaseModel):
    term: str = Field(description="Financial or economic concept name")
    definition: str = Field(description="Plain English definition")
    applied_to_news: str = Field(description="How it applies directly to this specific catalyst")

class EducationalInsights(BaseModel):
    historical_precedent: str = Field(description="Historical market precedent where a similar event occurred")
    retail_trap: str = Field(description="Common psychological or execution mistake beginner retail traders make")
    pop_quiz: str = Field(description="A 1-sentence diagnostic question to test comprehension of the concept")

class MarketEducationalReport(BaseModel):
    ticker: str = Field(description="Stock ticker or NONE")
    catalyst_summary: str = Field(description="Core event analysis and market expectations")
    company_impact: str = Field(description="Balance sheet, margin, or revenue implications")
    competitors_impact: str = Field(description="Read-across impact on sector peers")
    macro_ripple: str = Field(description="Macroeconomic context (rates, commodity prices, demand shifts)")
    key_terms: List[FinancialTerm] = Field(description="1-2 financial mechanisms illustrated by the news")
    educational_insights: EducationalInsights = Field(description="Precedent, traps, and quiz")
    trader_takeaway: str = Field(description="Actionable trade setup synthesizing fundamentals and technicals (or purely fundamental if technicals are unavailable)")


In [182]:
# identification_parser = PydanticOutputParser(pydantic_object=IdentifiedArticles)


In [183]:
identification_prompt = PromptTemplate(
    template="""You are an equity research associate covering the National Stock Exchange of India (NSE).
Review the following articles:

{articles_data}

Instructions:
1. For each article, determine if a specific, publicly listed Indian company is the primary subject.
2. If YES, set category to "SINGLE_STOCK" and output its clean NSE ticker (e.g., RELIANCE, INFY, TATAMOTORS, TVSMOTOR).
3. If the article is about macroeconomics, interest rates, GDP, or policy, set category to "MACRO_SECTOR" and ticker to "NONE".
4. If the article is non-financial noise (horoscopes, sports, entertainment, general lifestyle), set category to "NOISE" and ticker to "NONE".
5. Extract up to 5 relevant items.
""",
    input_variables=["articles_data"]
)

In [184]:
identification_model = model.with_structured_output(IdentifiedArticles)
identification_chain = identification_prompt | identification_model 

mentor_model = model.with_structured_output(MarketEducationalReport)


In [185]:
mentor_prompt = PromptTemplate(
    template="""You are a senior hedge fund portfolio manager and financial educator.
    Analyze this catalyst and provide structured educational mentorship.

    Catalyst Details:
    - Title: {title}
    - Source: {source}
    - Summary: {description}

    Market Info:
    - Ticker: {ticker} (CRITICAL: If 'NONE', treat this as a top-down macroeconomic, sectoral, or regulatory event).
    - Close Price: {close}
    - 14-Day RSI: {rsi}
    - 20-Day EMA: {ema20}
    - MACD: {macd}

    Analyze the event across these dimensions to populate the structured output:
    1. FIRST-ORDER EFFECT: Revenue vs margin impact, pricing power, or capacity changes.
    2. SECOND-ORDER EFFECT: Peer impact, market share dynamics, and supplier/buyer reactions.
    3. MACRO & SUPPLY CHAIN: Broader implications for the economy, interest rates, etc.
    4. CORE CONCEPTS TO MASTER: Extract 1-2 institutional financial terms (e.g., "Operating Leverage") and define them clearly.
    5. EDUCATIONAL INSIGHTS (Crucial): 
       - Provide a historical precedent (when has something similar happened before?).
       - Point out the "Retail Trap" (what amateur traders usually do wrong here).
       - Create a quick pop quiz question for the user to test their understanding.
    6. TRADER PLAYBOOK: Synthesize the news with the provided technicals. If technicals are 'N/A', provide a purely fundamental takeaway.
    """,
    input_variables=["title", "source", "description", "ticker", "close", "rsi", "ema20", "macd"]
)

In [186]:
mentor_chain = mentor_prompt | mentor_model


In [188]:
# def run_pipeline():
# Step 1: Ingest News
articles = fetch_free_market_news()

if not articles:
    print("\n[Pipeline Result] No news articles could be retrieved from the feed at this time. Pipeline finished.")
    # return

articles_text = "\n\n".join([
    f"Article {i+1}:\nTitle: {a['title']}\nSource: {a['source']}\nBody: {a['description']}"
    for i, a in enumerate(articles)
])

# Step 2: Map to NSE Equities
print("\n[Analysis] Extracting corporate catalysts and tickers...")
try:
    identified = identification_chain.invoke({"articles_data": articles_text})
except Exception as e:
    print(f"[Error] Failed to parse company extraction: {e}")
    # return

if not identified.items:
    print("\n[Pipeline Result] No actionable single-stock catalysts found in current headlines.")
    # return

# Step 3: Technicals + Educational Synthesis
final_reports: List[MarketEducationalReport] = []

for item in identified.items:
    # 1. Immediately drop noise (horoscopes, entertainment, sports)
    if item.category == "NOISE":
        continue

    # 2. Path A: Single-Stock Catalyst (Fundamentals + Technicals)
    if item.category == "SINGLE_STOCK" and item.ticker in VALID_NSE_TICKERS:
        tech_data = calculate_ticker_technicals(item.ticker)
        payload = {
            "mode": "SINGLE_STOCK",
            "title": item.news_title,
            "source": item.source,
            "description": item.description,
            "ticker": item.ticker,
            "close": f"₹{tech_data['close']}" if tech_data else "N/A",
            "rsi": str(tech_data["rsi"]) if tech_data else "N/A",
            "ema20": str(tech_data["ema20"]) if tech_data else "N/A",
            "macd": str(tech_data["macd"]) if tech_data else "N/A",
        }
        report = mentor_chain.invoke(payload)
        final_reports.append(report)

    # 3. Path B: Macroeconomic or Sectoral Catalyst (Pure Economic Mechanics)
    elif item.category == "MACRO_SECTOR":
        payload = {
            "title": item.news_title,
            "source": item.source,
            "description": item.description,
            "ticker": "NONE",  
            "close": "N/A",    
            "rsi": "N/A",      
            "ema20": "N/A",    
            "macd": "N/A"      
        }
        report = mentor_chain.invoke(payload)
        final_reports.append(report)
        
# Step 4: Display Output
print(f"\n{'='*70}\nGENERATED {len(final_reports)} MARKET INTELLIGENCE CASE STUDIES\n{'='*70}")
for idx, r in enumerate(final_reports, 1):
    print(f"\nCASE STUDY #{idx}: [{r.ticker}]")
    print(f"Catalyst: {r.catalyst_summary}")
    print(f"First-Order Impact: {r.company_impact}")
    print(f"Read-Across (Peers): {r.competitors_impact}")
    print(f"Macro Link: {r.macro_ripple}")
    
    print("\n🎓 Classroom Concepts:")
    for term in r.key_terms:
        print(f"  • {term.term}: {term.definition}")
        print(f"    Applied: {term.applied_to_news}")
        
    print(f"\n🏛️ Historical Precedent:\n  {r.educational_insights.historical_precedent}")
    print(f"⚠️ The Retail Trap:\n  {r.educational_insights.retail_trap}")
    print(f"❓ Pop Quiz:\n  {r.educational_insights.pop_quiz}")
    print(f"\n💼 Trader Playbook:\n  {r.trader_takeaway}")
    print("-" * 70)



[Analysis] Extracting corporate catalysts and tickers...


KeyboardInterrupt: 

In [190]:
print(identified)
print(len(identified.items))


items=[ExtractedEntity(category='SINGLE_STOCK', ticker='EPIGRAL', news_title='Epigral Standalone March 2024 Net Sales at Rs 524.63 crore, down 6.68% Y-o-Y', description="Epigral's standalone net sales for March 2024 were Rs 524.63 crore, marking a 6.68% year-over-year decline.", source='Unknown'), ExtractedEntity(category='SINGLE_STOCK', ticker='HATSUNAGRO', news_title='Hatsun Agro Standalone March 2024 Net Sales at Rs 2,046.87 crore, up 14.38% Y-o-Y', description='Hatsun Agro reported standalone net sales of Rs 2,046.87 crore for March 2024, showing a 14.38% year-over-year increase.', source='Unknown'), ExtractedEntity(category='SINGLE_STOCK', ticker='TANFACIND', news_title='Tanfac Ind Standalone March 2024 Net Sales at Rs 102.81 crore, down 11.01% Y-o-Y', description="Tanfac Ind's standalone net sales for March 2024 were Rs 102.81 crore, a decrease of 11.01% year-over-year.", source='Unknown'), ExtractedEntity(category='SINGLE_STOCK', ticker='TEJASNET', news_title='Tejas Networks Sta

In [ ]:
# if __name__ == "__main__":
#     run_pipeline()


[Analysis] Extracting corporate catalysts and tickers...
[Error] Failed to parse company extraction: Failed to parse IdentifiedArticles from completion {"items": [{"category": "SINGLE_STOCK", "ticker": "RELIANCE", "news_title": "Reliance Standalone March 2024 Net Sales at Rs 146,832.00 crore, up 23.7% Y-o-Y", "description": "Reliance Standalone March 2024 Net Sales at Rs"}]}. Got: 1 validation error for IdentifiedArticles
items.0.source
  Field required [type=missing, input_value={'category': 'SINGLE_STOC...h 2024 Net Sales at Rs'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
